In [1]:

import os
import pandas as pd
import numpy as np
import scanpy as sc
import scipy.sparse as sp  # 可以从这个命名空间中直接使用 csr_matrix 和 issparse
from cmapPy.pandasGEXpress.parse_gctx import parse
import rdkit
from rdkit.Chem import AllChem
from rdkit import Chem
from tqdm import tqdm
import random
import torch 
from torch.utils.data import Dataset

import h5py


import warnings
warnings.filterwarnings('ignore')

def create_sample_mapping(filtered_info_cp, info_vehicle):
    """we want to map each sample_id in filtered_info_cp to a random sample_id in info_vehicle with the same det_plate
        det plate have plate info and cell line info, so we can use it to match the cell line"""
    vehicle_grouped = info_vehicle.groupby('det_plate')
    result_dict = {}
    
    for _, row in filtered_info_cp.iterrows():
        det_plate = row['det_plate']
        sample_id = row['sample_id']
        
        if det_plate in vehicle_grouped.groups:
            matching_rows = vehicle_grouped.get_group(det_plate)
            random_row = matching_rows.sample(n=1).iloc[0]
            result_dict[sample_id] = random_row['sample_id']
    
    return result_dict



#load datas
print("Loading data...")
info = pd.read_csv('~/cbfp_vae/dataprocess/data/instinfo_beta.txt', sep='\t',header=0,low_memory=False)
info_qp = info[info['qc_pass'] == 1]
info_cp = info_qp[info_qp['pert_type'] == 'trt_cp'] #get the chemical perturbation only
info_vehicle = info_qp[info_qp['pert_type'] == 'ctl_vehicle']
cell_info = pd.read_csv('~/cbfp_vae/dataprocess/data/cellinfo_beta.txt', sep='\t',header=0,)# get cell info

#wcnm comp info have repeat pert id because some compound have multiple target but we only need smiles
# so we drop duplicates here
comp_info = pd.read_csv('~/cbfp_vae/dataprocess/data/compoundinfo_beta.txt', sep='\t',header=0)# get compound info
print("Total compounds:", comp_info.shape[0])
comp_info_unique = comp_info.drop_duplicates(subset=['pert_id'], keep='first')

comp_info_unique_clean = comp_info_unique.dropna(subset=['canonical_smiles'])
print("Unique compounds:", comp_info_unique_clean.shape[0])
gen_info = pd.read_csv('~/cbfp_vae/dataprocess/data/geneinfo_beta.txt', sep='\t',header=0,)# get gene info
my_ds = parse("~/cbfp_vae/dataprocess/data/level3_beta_trt_cp_n1805898x12328.gctx") # trt
cl_ds = parse("~/cbfp_vae/dataprocess/data/level3_beta_ctl_n188708x12328.gctx") # ctl

print("Data loaded!")

print("Merging compound and cell info...")


original_rows = len(info_cp)
print(f"oringinal info_cp: {original_rows}")

info_cp = info_cp.merge(comp_info_unique_clean, how='left', on='pert_id')
info_cp = info_cp.merge(cell_info, how='left', on='cell_iname')

print("Compound and cell info merged!")
print(f'checking {info_cp.shape[0]} samples and {info_cp.shape[1]} features in total')

filtered_info_cp = info_cp[(info_cp['pert_time'] == 24) & (info_cp['pert_dose'] == 10)]

##remove invalid smiles
filtered_info_cp = filtered_info_cp.dropna(subset=['canonical_smiles'])
filtered_info_cp = filtered_info_cp[filtered_info_cp['canonical_smiles'] != 'restricted']
vc = filtered_info_cp["pert_id"].value_counts()
keep_ids = vc[vc >= 10].index
filtered_info_cp = filtered_info_cp[filtered_info_cp["pert_id"].isin(keep_ids)].copy()
print(filtered_info_cp.shape)
sample_mapping = create_sample_mapping(filtered_info_cp, info_vehicle) 
print(f"we get {len(sample_mapping)} pairs of samples")

sample_to_smiles = dict(zip(filtered_info_cp['sample_id'], filtered_info_cp['canonical_smiles']))

canonical_smiles = [sample_to_smiles[sample_id] for sample_id in sample_mapping.keys()]
print("Encoding compound SMILES to fingerprints using Signaturizer...")


gene_info_ctl = [my_ds.data_df[i] for i in sample_mapping.keys()]
gene_info_trt = [
    (my_ds.data_df[i] if i in my_ds.data_df.columns 
    else cl_ds.data_df[i].tolist())
    for i in sample_mapping.values()
    ]
dose_list = [10] * len(canonical_smiles)   

print("Gene expression data prepared.")




/home/liuxiaoping/miniconda3/envs/prnet/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...
Total compounds: 39321
Unique compounds: 28634
Data loaded!
Merging compound and cell info...
oringinal info_cp: 1312170
Compound and cell info merged!
checking 1312170 samples and 55 features in total
(210485, 55)
we get 208672 pairs of samples
Encoding compound SMILES to fingerprints using Signaturizer...
Gene expression data prepared.


In [3]:
import  numpy as np
arr = np.load("./test_indicesall.npy")   # 绝大多数情况这样就够了
print(type(arr))
print(arr.shape, arr.dtype)
print(arr[:5])    

<class 'numpy.ndarray'>
(5982,) int64
[  943 27987 34655 35229 30639]


In [4]:
keys = set(sample_mapping.keys())

filtered_info_cp_new = filtered_info_cp[filtered_info_cp["sample_id"].isin(keys)].copy()
filtered_info_cp_new

,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_itime,pert_time_unit,cell_mfc_name,pert_mfc_id,...,donor_ethnicity,donor_sex,donor_tumor_phase,cell_lineage,primary_disease,subtype,provider_name,growth_pattern,ccle_name,cell_alias
118734,b10,10.0,10.0,uM,10 uM,24.0,24 h,h,PC3,BRD-K81418486,...,Caucasian,M,Metastatic,prostate,prostate cancer,adenocarcinoma,ATCC,mix,PC3_PROSTATE,PC.3|PC-3
118761,b11,10.0,10.0,uM,10 uM,24.0,24 h,h,A375,BRD-A19500257,...,Unknown,F,Metastatic,skin,skin cancer,melanoma,ATCC,adherent,A375_SKIN,A 375|A-375
118771,b10,10.0,10.0,uM,10 uM,24.0,24 h,h,NKDBA,BRD-K79759585-001-02-1,...,Unknown,Unknown,Unknown,unknown,unknown,unknown,NaN,unknown,NaN,NaN
118790,b11,10.0,10.0,uM,10 uM,24.0,24 h,h,PC3,BRD-K09068728,...,Caucasian,M,Metastatic,prostate,prostate cancer,adenocarcinoma,ATCC,mix,PC3_PROSTATE,PC.3|PC-3
118794,b10,10.0,10.0,uM,10 uM,24.0,24 h,h,MCF7,BRD-A47829399,...,Caucasian,F,Metastatic,breast,breast cancer,adenocarcinoma,ATCC,adherent,MCF7_BREAST,IBMF-7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
597965,f2b4,10.0,10.0,uM,10 uM,24.0,24 h,h,PC3,BRD-K94379058-001-03-5,...,Caucasian,M,Metastatic,prostate,prostate cancer,adenocarcinoma,ATCC,mix,PC3_PROSTATE,PC.3|PC-3
597966,f2b4,10.0,10.0,uM,10 uM,24.0,24 h,h,PC3,BRD-K62353524-001-01-8,...,Caucasian,M,Metastatic,prostate,prostate cancer,adenocarcinoma,ATCC,mix,PC3_PROSTATE,PC.3|PC-3
597972,f2b5,10.0,10.0,uM,10 uM,24.0,24 h,h,MCF7,BRD-A41692738-001-03-5,...,Caucasian,F,Metastatic,breast,breast cancer,adenocarcinoma,ATCC,adherent,MCF7_BREAST,IBMF-7
597973,f2b5,10.0,10.0,uM,10 uM,24.0,24 h,h,MDAMB231,BRD-K56277358,...,Caucasian,F,Metastatic,breast,breast cancer,adenocarcinoma,ATCC,adherent,MDAMB231_BREAST,MB231|MDA-MB-231


In [8]:
n_cell = filtered_info_cp_new["primary_disease"].nunique(dropna=True)
n_pert = filtered_info_cp_new["pert_id"].nunique(dropna=True)

n_cell, n_pert

(27, 6316)

####为细胞系做组织的划分

In [10]:
import os
import numpy as np
from sklearn.model_selection import GroupKFold


def split_5fold_train_val_by_group(
    df,
    group_col,
    save_dir,
    file_suffix="disease",
    n_splits=5,
):
    idx = np.arange(len(df))
    groups = df[group_col].to_numpy()

    os.makedirs(save_dir, exist_ok=True)

    gkf = GroupKFold(n_splits=n_splits)

    all_splits = []

    for fold, (train_idx, val_idx) in enumerate(
        gkf.split(idx, groups=groups),
        start=1
    ):

        # 文件命名
        train_idx_path = os.path.join(
            save_dir,
            f"fold_{fold}_train_indices_{file_suffix}.npy"
        )

        val_idx_path = os.path.join(
            save_dir,
            f"fold_{fold}_val_indices_{file_suffix}.npy"
        )

        # 保存
        np.save(train_idx_path, train_idx.astype(np.int64))
        np.save(val_idx_path, val_idx.astype(np.int64))

        # 检查 group 是否泄漏
        s_tr = set(df.iloc[train_idx][group_col])
        s_va = set(df.iloc[val_idx][group_col])

        print(f"\n===== Fold {fold} =====")
        print(f"group_col={group_col}")
        print("sizes:", len(train_idx), len(val_idx))
        print(
            "ratios:",
            len(train_idx) / len(df),
            len(val_idx) / len(df),
        )
        print("train-val group overlap:", len(s_tr & s_va))

        all_splits.append((train_idx, val_idx))

    return all_splits
###cell_mfc_name pert_id primary_disease
all_splits = split_5fold_train_val_by_group(
    filtered_info_cp_new,
    group_col="primary_disease",
    save_dir="./diseasecv5_indices",
    file_suffix="disease",
    n_splits=5,
)


===== Fold 1 =====
group_col=primary_disease
sizes: 163379 45293
ratios: 0.7829464422634566 0.21705355773654347
train-val group overlap: 0

===== Fold 2 =====
group_col=primary_disease
sizes: 167799 40873
ratios: 0.8041280095077442 0.1958719904922558
train-val group overlap: 0

===== Fold 3 =====
group_col=primary_disease
sizes: 167764 40908
ratios: 0.8039602821653121 0.19603971783468793
train-val group overlap: 0

===== Fold 4 =====
group_col=primary_disease
sizes: 167835 40837
ratios: 0.8043005290599601 0.19569947094003987
train-val group overlap: 0

===== Fold 5 =====
group_col=primary_disease
sizes: 167911 40761
ratios: 0.804664737003527 0.19533526299647294
train-val group overlap: 0


###为化合物做划分

数据集划分的关键！！！！

In [6]:
arr = np.load("./test_indices_ctl2trt.npy", allow_pickle=False)  # 一般不需要改
print(type(arr), arr.shape, arr.dtype)
arr

<class 'numpy.ndarray'> (31301,) int64


array([125168,  85164, 120051, ..., 195292, 196848, 145640])